# RESuM paper reproduction with RESUM-FLEX

This notebook replays the reproducible parts of [RESuM: A Rare Event Surrogate Model](https://arxiv.org/abs/2410.03873) using only code and data bundled with RESUM-FLEX. It neither imports nor reads from the original RESuM checkout. It deliberately distinguishes three evidence levels:

- **Exact recomputation:** a result can be rebuilt from event-level inputs and a declared model configuration.
- **Aggregate replay:** published intermediate CSV values are loaded and passed through the new framework.
- **Unavailable:** required event data, checkpoints, or exact historical state are absent.

The bundled aggregate artifacts support an MFGP replay and validation study. They do not support an end-to-end CNP retraining or an exact recreation of every paper figure.

The design vector is $\boldsymbol{\theta}=(r,d,n,\phi,L)$: radius, panel thickness, number of panels, panel angle, and panel length. For a trial with $N$ simulated events and $m$ rare events, the elementary estimator is

$$y(\boldsymbol{\theta})=\frac{m}{N}. $$

The bundled CSV additionally contains the experiment-specific converted $^{77(m)}\mathrm{Ge}$ production rate $R(\boldsymbol{\theta})$, which is the final physical target in this replay. This is a **practice-truth** workflow: truth is supplied by held-out physical simulations. A **theory-truth** workflow instead supplies a known analytical probability $t(\boldsymbol{\theta},\boldsymbol{\varphi})$ and can compare a model directly with that function. No analytical theory truth exists for this detector study, so the two meanings are not mixed here.

## 1. Setup and data provenance

The two input CSV files live under `notebooks/data/paper_2410_03873/`; their source commit, original paths, and checksums are documented beside the data. Keeping these small aggregate fixtures in RESUM-FLEX makes this notebook self-contained with respect to the old implementation. The notebook uses no HDF5 input, so `h5py` is not required. Install the GP extra before running: `uv sync --extra gp`.

The first code cell imports only the Python standard library, NumPy, Matplotlib, and the new `core.scaling` and `core.surrogate_mfgp` implementations. It resolves all paths relative to the RESUM-FLEX repository and fixes the NumPy random seed. `RESUM_GP_RESTARTS` controls the number of independent marginal-likelihood optimizations; the default is 10. The second code cell reads numeric columns with `csv.DictReader`, gives unit-bearing legacy columns unambiguous internal names, constructs the matrix $X=[\boldsymbol{\theta}_1,\ldots,\boldsymbol{\theta}_B]^T$, and declares the physical design bounds. No model is fitted during data loading.

In [ ]:
from __future__ import annotations

import csv
import os
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

project_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(path for path in project_candidates if (path / "core").is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from core.optimizer import BoxBounds, ExpectedImprovementAcquisition
from core.scaling import MinMaxScaler
from core.surrogate_mfgp import MultiFidelityGP

DATA_ROOT = PROJECT_ROOT / "notebooks" / "data" / "paper_2410_03873"
TRAIN_PATH = DATA_ROOT / "cnp_v1.6_output.csv"
VALIDATION_PATH = DATA_ROOT / "hf_validation_data_v1.2.csv"

missing = [path for path in (TRAIN_PATH, VALIDATION_PATH) if not path.is_file()]
if missing:
    raise FileNotFoundError(
        f"Missing bundled aggregate reproduction data: {missing}"
    )

SEED = 1729
N_RESTARTS = int(os.environ.get("RESUM_GP_RESTARTS", "10"))
np.random.seed(SEED)
warnings.filterwarnings("ignore", category=RuntimeWarning, module="GPy")
print(f"Bundled data root: {DATA_ROOT}")
print(f"GP optimizer restarts: {N_RESTARTS}")

In [ ]:
THETA_NAMES = ("radius", "thickness", "npanels", "theta", "length")
THETA_LABELS = (
    "Radius [cm]",
    "Thickness [cm]",
    "Number of panels",
    "Panel angle [deg]",
    "Length [cm]",
)
THETA_LOW = np.array([0.0, 0.0, 0.0, 0.0, 0.0])
THETA_HIGH = np.array([265.0, 20.0, 360.0, 90.0, 150.0])
THETA_FIXED = np.array([160.0, 2.0, 40.0, 45.0, 20.0])
theta_scaler = MinMaxScaler.from_bounds(THETA_LOW, THETA_HIGH)


def read_numeric_csv(path: Path, columns: dict[str, str]) -> dict[str, np.ndarray]:
    with path.open(newline="", encoding="utf-8") as stream:
        rows = list(csv.DictReader(stream))
    return {
        output_name: np.asarray([float(row[input_name]) for row in rows])
        for output_name, input_name in columns.items()
    }


train = read_numeric_csv(
    TRAIN_PATH,
    {
        "iteration": "iteration",
        "fidelity": "fidelity",
        "n_samples": "n_samples",
        "radius": "radius",
        "thickness": "thickness",
        "npanels": "npanels",
        "theta": "theta",
        "length": "length",
        "event_count": "y_raw",
        "y_cnp": "y_cnp",
        "y_cnp_error": "y_cnp_err",
        "physical_rate": "y_rGe77[nuc/(kg*yr)]",
    },
)
validation = read_numeric_csv(
    VALIDATION_PATH,
    {
        "fidelity": "Fidelity",
        "radius": "Radius[cm]",
        "thickness": "Thickness[cm]",
        "npanels": "NPanels",
        "theta": "Theta[deg]",
        "length": "Length[cm]",
        "event_count": "y_raw",
        "y_cnp": "y_cnp",
        "y_cnp_error": "y_cnp_err",
        "physical_rate": "y_rGe77[nuc/(kg*yr)]",
    },
)


def theta_matrix(data: dict[str, np.ndarray]) -> np.ndarray:
    return np.column_stack([data[name] for name in THETA_NAMES])


X_train = theta_matrix(train)
X_validation = theta_matrix(validation)

## 2. Artifact audit

The paper reports 304 LF and 4 HF initial trials, followed by six paired LF/HF additions for a final total of 310 LF and 10 HF trials. For stored active-learning iteration $k$, the code forms

$$\mathcal{D}^{(k)}=\{i:\operatorname{iteration}_i\leq k\}$$

and separately counts rows with fidelity labels 0 (LF) and 1 (HF). This is a data-integrity check, not a statistical calculation. The bundled aggregate CSV contains one fewer LF row at every stage: 303 LF initially and 309 LF finally. The independent HF validation set contains all 100 reported trials. This mismatch prevents an exact numerical claim even before model-version differences are considered.

In [ ]:
print("iteration | LF rows | HF rows")
print("----------+---------+--------")
for iteration in range(7):
    included = train["iteration"] <= iteration
    n_lf = int(np.sum(included & (train["fidelity"] == 0)))
    n_hf = int(np.sum(included & (train["fidelity"] == 1)))
    print(f"{iteration:9d} | {n_lf:7d} | {n_hf:7d}")

assert len(validation["physical_rate"]) == 100
assert int(np.sum(train["fidelity"] == 1)) == 10
print(f"Validation rows: {len(validation['physical_rate'])}")
print("Paper final LF total: 310; available aggregate LF total: 309")

## 3. CNP retraining placeholder — unavailable

A CNP would receive a design $\boldsymbol{\theta}$, per-event features $\boldsymbol{\varphi}_i$, and binary observations $x_i\in\{0,1\}$. An encoder maps context events to representations $\mathbf{h}_i$, an aggregation produces $\mathbf{r}=N_C^{-1}\sum_{i\in C}\mathbf{h}_i$, and a decoder predicts event probabilities

$$\beta_i=p(x_i=1\mid\boldsymbol{\theta},\boldsymbol{\varphi}_i,\mathbf{r}).$$

The trial-level denoised statistic used by the MFGP is

$$\bar{\beta}(\boldsymbol{\theta})=\frac{1}{N}\sum_{i=1}^{N}\beta_i.$$

Exact recomputation therefore requires the event features, labels, context split, network configuration, learned parameters, and random state. The available source history does not provide a complete portable LF/HF event corpus or the current referenced checkpoint. The following intentionally empty cell is reserved for a future CNP retraining run once those inputs and a declared checkpoint/configuration contract are available.

Downstream cells use the checked-in `y_cnp` aggregate values. They are therefore aggregate replay, not CNP recomputation.

## 4. Aggregate CNP diagnostic replay

The bundled `y_raw` column is the count $m$, while `physical_rate` is the paper's final physics target $R$ in nuclei/(kg·yr). For each coordinate $\theta_j$, the code selects the initial LF subset, sorts rows by $\theta_j$, and scatters the stored pairs $(\theta_j,R)$ and $(\theta_j,\bar{\beta}_{LF})$. No interpolation, normalization, or fit is applied in this step. The plots compare two stored aggregates only to inspect whether they exhibit related design dependence; they do not claim that $R$ and $\bar{\beta}$ have identical units.

In [ ]:
initial_lf = (train["iteration"] == 0) & (train["fidelity"] == 0)
figure, axes = plt.subplots(2, 3, figsize=(15, 8))
for index, (name, label) in enumerate(zip(THETA_NAMES, THETA_LABELS, strict=True)):
    axis = axes.flat[index]
    order = np.argsort(train[name][initial_lf])
    x = train[name][initial_lf][order]
    axis.scatter(
        x, train["physical_rate"][initial_lf][order], s=13, alpha=0.65,
        label="LF physical rate",
    )
    axis.scatter(
        x, train["y_cnp"][initial_lf][order], s=13, alpha=0.65,
        label="LF CNP aggregate",
    )
    axis.set_xlabel(label)
    axis.set_ylabel("Stored aggregate value")
    axis.grid(alpha=0.2)
axes.flat[0].legend()
axes.flat[-1].axis("off")
figure.suptitle("Initial LF aggregate diagnostics (stored CNP outputs)")
figure.tight_layout()

## 5. Three-fidelity MFGP replay

RESUM-FLEX assigns the aggregate artifacts to three mathematical levels:

1. `f0`: LF CNP aggregate, an event-level denoised statistic from low-fidelity trials.
2. `f1`: HF CNP aggregate, the same statistic evaluated on high-fidelity trials.
3. `f2`: HF physical rate, the final design objective predicted in nuclei/(kg·yr).

Each input coordinate is first mapped from its declared physical interval $[a_j,b_j]$ to $[-1,1]$:

$$s_j(\theta_j)=-1+2\frac{\theta_j-a_j}{b_j-a_j}. $$

This prevents radius or panel count from dominating distance calculations merely because their numerical ranges are larger. The linear autoregressive multi-fidelity model is

$$f_{i+1}(\mathbf{s})=\rho_i f_i(\mathbf{s})+\delta_i(\mathbf{s}),$$

where each discrepancy $\delta_i$ is an independent Gaussian process and $\rho_i$ is learned. Each GP uses an isotropic RBF covariance

$$k(\mathbf{s},\mathbf{s}')=\sigma_f^2\exp\!\left(-\frac{\lVert\mathbf{s}-\mathbf{s}'\rVert^2}{2\ell^2}\right).$$

The paper used one length scale per fidelity, so this replay selects `ard=False`. GPy learns kernel, cross-fidelity scale, and likelihood parameters by maximizing the log marginal likelihood; 10 random restarts are used by default to reduce sensitivity to local optima. At replay iteration $k$, only rows in $\mathcal{D}^{(k)}$ are passed to the new `MultiFidelityGP.fit` method.

In [ ]:
def fit_resum_model(max_iteration: int) -> MultiFidelityGP:
    included = train["iteration"] <= max_iteration
    lf = included & (train["fidelity"] == 0)
    hf = included & (train["fidelity"] == 1)
    X_lf = theta_scaler.transform(X_train[lf])
    X_hf = theta_scaler.transform(X_train[hf])
    model = MultiFidelityGP(3, len(THETA_NAMES), kernel="rbf", ard=False)
    np.random.seed(SEED + max_iteration)
    return model.fit(
        [X_lf, X_hf, X_hf],
        [
            train["y_cnp"][lf, None],
            train["y_cnp"][hf, None],
            train["physical_rate"][hf, None],
        ],
        n_restarts=N_RESTARTS,
    )


resum_models = []
for iteration in range(7):
    print(f"Fitting aggregate replay through iteration {iteration}...")
    resum_models.append(fit_resum_model(iteration))
print("Finished fitting seven sequential surrogate states.")

### Sequential one-dimensional projections

For coordinate $j$, the projection query varies only that coordinate while holding the others at the paper's fixed vector $\boldsymbol{\theta}_{fixed}=(160,2,40,45,20)$. In symbols,

$$\mathbf{q}_j(z)=(\theta_{fixed,1},\ldots,z,\ldots,\theta_{fixed,5}).$$

For each stored iteration $k=0,\ldots,6$, the code plots the highest-fidelity posterior mean $\mu_2^{(k)}(\mathbf{q}_j(z))$. The shaded final band is $\mu_2^{(6)}\pm\sigma_2^{(6)}$, with $\sigma=\sqrt{\operatorname{Var}[f_2\mid\mathcal{D}^{(6)}]}$. These projections replay the model update after each stored active-learning iteration. They are analogous to Figure 4, but cannot reproduce its exact curves because the exact historical fitted state and complete initial LF set are unavailable.

In [ ]:
figure, axes = plt.subplots(2, 3, figsize=(16, 8))
iteration_colors = plt.cm.Oranges(np.linspace(0.25, 0.95, 7))
for dimension, label in enumerate(THETA_LABELS):
    axis = axes.flat[dimension]
    grid = np.linspace(THETA_LOW[dimension], THETA_HIGH[dimension], 180)
    query = np.repeat(THETA_FIXED[None, :], len(grid), axis=0)
    query[:, dimension] = grid
    query_scaled = theta_scaler.transform(query)
    for iteration, (model, color) in enumerate(zip(resum_models, iteration_colors)):
        mean, variance = model.predict(query_scaled, fidelity=2)
        axis.plot(grid, mean, color=color, linewidth=1.2, label=f"Iteration {iteration}")
        if iteration == 6:
            sigma = np.sqrt(variance)
            axis.fill_between(grid, mean - sigma, mean + sigma, color=color, alpha=0.2)
    axis.set_xlabel(label)
    axis.set_ylabel("HF physical rate")
    axis.grid(alpha=0.2)
axes.flat[-1].axis("off")
axes.flat[0].legend(ncol=2, fontsize=8)
figure.suptitle("Sequential highest-fidelity projections")
figure.tight_layout()

### Final cross-fidelity projections

The preceding plot isolates how the final physical prediction changes over iterations. The next plot instead fixes iteration 6 and compares the posterior means and one-standard-deviation bands of $f_0$, $f_1$, and $f_2$. It makes the learned autoregressive bridge visible: information from dense LF CNP aggregates enters $f_1$ through $\rho_0$, and information from sparse HF CNP aggregates enters the physical target $f_2$ through $\rho_1$. Because the CNP aggregates and physical rate have different meanings, their shared axis is for structural comparison rather than unit equality.

In [ ]:
fidelity_labels = ("LF CNP aggregate", "HF CNP aggregate", "HF physical rate")
fidelity_colors = ("#26c6da", "#00838f", "#ef6c00")
final_model = resum_models[-1]
figure, axes = plt.subplots(2, 3, figsize=(16, 8))
for dimension, label in enumerate(THETA_LABELS):
    axis = axes.flat[dimension]
    grid = np.linspace(THETA_LOW[dimension], THETA_HIGH[dimension], 180)
    query = np.repeat(THETA_FIXED[None, :], len(grid), axis=0)
    query[:, dimension] = grid
    query_scaled = theta_scaler.transform(query)
    for fidelity, (fidelity_label, color) in enumerate(
        zip(fidelity_labels, fidelity_colors, strict=True)
    ):
        mean, variance = final_model.predict(query_scaled, fidelity=fidelity)
        sigma = np.sqrt(variance)
        axis.plot(grid, mean, color=color, label=fidelity_label)
        axis.fill_between(grid, mean - sigma, mean + sigma, color=color, alpha=0.12)
    axis.set_xlabel(label)
    axis.set_ylabel("Stored aggregate value")
    axis.grid(alpha=0.2)
axes.flat[-1].axis("off")
axes.flat[0].legend(fontsize=8)
figure.suptitle("Final cross-fidelity posterior projections")
figure.tight_layout()

### Expected-improvement replay

For minimization, let $R_{best}^{(k)}$ be the smallest observed HF physical rate through iteration $k$. At candidate $\boldsymbol{\theta}$, define $z=(R_{best}^{(k)}-\mu_2^{(k)})/\sigma_2^{(k)}$. Expected improvement is

$$\operatorname{EI}^{(k)}(\boldsymbol{\theta})=(R_{best}^{(k)}-\mu_2^{(k)}(\boldsymbol{\theta}))\Phi(z)+\sigma_2^{(k)}(\boldsymbol{\theta})\phi(z),$$

where $\Phi$ and $\phi$ are the standard-normal CDF and PDF. The cell evaluates the new framework's `ExpectedImprovementAcquisition` along the radius projection $\mathbf{q}_{radius}(r)$. This reproduces the acquisition definition and its sequential evaluation, not the paper's exact historical surface or selected point: physical geometry constraints, the missing LF row, and the original fitted state are not recoverable in full.

In [ ]:
radius_grid = np.linspace(THETA_LOW[0], THETA_HIGH[0], 300)
radius_query = np.repeat(THETA_FIXED[None, :], len(radius_grid), axis=0)
radius_query[:, 0] = radius_grid
radius_query_scaled = theta_scaler.transform(radius_query)
scaled_bounds = BoxBounds(low=-np.ones(5), high=np.ones(5))

figure, axis = plt.subplots(figsize=(10, 4))
for iteration, (model, color) in enumerate(zip(resum_models, iteration_colors)):
    hf = (train["iteration"] <= iteration) & (train["fidelity"] == 1)
    acquisition = ExpectedImprovementAcquisition(
        model,
        scaled_bounds,
        incumbent=float(np.min(train["physical_rate"][hf])),
        target="min",
        fidelity=2,
    )
    score = acquisition.score(radius_query_scaled)
    axis.plot(radius_grid, score, color=color, label=f"Iteration {iteration}")
axis.set_xlabel("Radius [cm]")
axis.set_ylabel("Expected improvement")
axis.set_title("Sequential radius acquisition replay")
axis.legend(ncol=2, fontsize=8)
axis.grid(alpha=0.2)
figure.tight_layout()

## 6. Independent HF validation replay

For each held-out HF design $\boldsymbol{\theta}_i$, the final model returns posterior mean $\mu_i$ and variance $v_i$; the standard deviation is $\sigma_i=\sqrt{v_i}$. Empirical coverage at level $q\in\{1,2,3\}$ is

$$C_q=\frac{100}{N_{val}}\sum_{i=1}^{N_{val}}\mathbf{1}\left[|R_i-\mu_i|\leq q\sigma_i\right].$$

The paper reports $(C_1,C_2,C_3)=(69\%,95\%,100\%)$. The code computes exactly this statistic on the 100 bundled practice-truth HF trials and prints paper and replay values side by side. Sorting by posterior mean affects only plot readability, not coverage. Differences are expected and must not be hidden: the aggregate file is missing one LF trial, the exact historical trained state is unavailable, and the refactored implementation cannot recreate undeclared optimizer state.

In [ ]:
def coverage_percentages(
    model: MultiFidelityGP,
    X: np.ndarray,
    target: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, dict[int, float]]:
    mean, variance = model.predict(theta_scaler.transform(X))
    sigma = np.sqrt(variance)
    coverage = {
        level: 100.0 * float(np.mean(np.abs(target - mean) <= level * sigma))
        for level in (1, 2, 3)
    }
    return mean, sigma, coverage


validation_target = validation["physical_rate"]
resum_mean, resum_sigma, resum_coverage = coverage_percentages(
    resum_models[-1], X_validation, validation_target
)
print("Coverage (%) | 1 sigma | 2 sigma | 3 sigma")
print("-------------+---------+---------+--------")
print("Paper        |    69.0 |    95.0 |   100.0")
print(
    "Replay       | "
    + " | ".join(f"{resum_coverage[level]:7.1f}" for level in (1, 2, 3))
)

order = np.argsort(resum_mean)
trial = np.arange(len(order))
figure, axis = plt.subplots(figsize=(14, 5))
colors = {3: "#ef9a9a", 2: "#ffe082", 1: "#a5d6a7"}
for level in (3, 2, 1):
    axis.fill_between(
        trial,
        resum_mean[order] - level * resum_sigma[order],
        resum_mean[order] + level * resum_sigma[order],
        color=colors[level],
        label=f"{level} sigma band",
    )
axis.plot(trial, resum_mean[order], color="tab:orange", label="Posterior mean")
axis.scatter(trial, validation_target[order], s=16, color="black", label="HF validation")
axis.set_xlabel("Validation trial, sorted by posterior mean")
axis.set_ylabel("Physical rate [nuclei/(kg yr)]")
axis.set_title("RESUM-FLEX aggregate replay coverage")
axis.legend(ncol=5, fontsize=8)
axis.grid(alpha=0.2)
figure.tight_layout()

## 7. No-CNP ablation replay

The ablation removes both denoised CNP levels. It defines only

$$g_0(\boldsymbol{\theta})=R_{LF}(\boldsymbol{\theta}),\qquad g_1(\boldsymbol{\theta})=R_{HF}(\boldsymbol{\theta}),$$

with $g_1=\rho g_0+\delta$. Fitting the same isotropic RBF family and evaluating the same $C_q$ isolates the structural effect of including $\bar{\beta}_{LF}$ and $\bar{\beta}_{HF}$; it does not isolate implementation or optimizer changes. The paper reports $(12\%,24\%,47\%)$. This cell performs that structural ablation with the new implementation and prints both sets of values. It is a method replay, not an exact numerical reproduction, for the same artifact and model-state reasons listed above.

In [ ]:
def fit_no_cnp_model() -> MultiFidelityGP:
    lf = train["fidelity"] == 0
    hf = train["fidelity"] == 1
    X_lf = theta_scaler.transform(X_train[lf])
    X_hf = theta_scaler.transform(X_train[hf])
    model = MultiFidelityGP(2, len(THETA_NAMES), kernel="rbf", ard=False)
    np.random.seed(SEED + 100)
    return model.fit(
        [X_lf, X_hf],
        [
            train["physical_rate"][lf, None],
            train["physical_rate"][hf, None],
        ],
        n_restarts=N_RESTARTS,
    )


no_cnp_model = fit_no_cnp_model()
no_cnp_mean, no_cnp_sigma, no_cnp_coverage = coverage_percentages(
    no_cnp_model, X_validation, validation_target
)
print("Coverage (%) | 1 sigma | 2 sigma | 3 sigma")
print("-------------+---------+---------+--------")
print("Paper        |    12.0 |    24.0 |    47.0")
print(
    "Replay       | "
    + " | ".join(f"{no_cnp_coverage[level]:7.1f}" for level in (1, 2, 3))
)

order = np.argsort(no_cnp_mean)
trial = np.arange(len(order))
figure, axis = plt.subplots(figsize=(14, 5))
for level in (3, 2, 1):
    axis.fill_between(
        trial,
        no_cnp_mean[order] - level * no_cnp_sigma[order],
        no_cnp_mean[order] + level * no_cnp_sigma[order],
        color=colors[level],
        label=f"{level} sigma band",
    )
axis.plot(trial, no_cnp_mean[order], color="tab:blue", label="Posterior mean")
axis.scatter(trial, validation_target[order], s=16, color="black", label="HF validation")
axis.set_xlabel("Validation trial, sorted by posterior mean")
axis.set_ylabel("Physical rate [nuclei/(kg yr)]")
axis.set_title("No-CNP two-fidelity ablation replay")
axis.legend(ncol=5, fontsize=8)
axis.grid(alpha=0.2)
figure.tight_layout()

## 8. Reproduction boundary

| Paper result | Status in this notebook | Reason |
|---|---|---|
| Initial and sequential design samples | Aggregate replay | Stored CSV rows are available, but one initial LF row is missing |
| CNP training and event-level diagnostics | Unavailable | Complete LF/HF event corpus and current checkpoint are absent |
| Sequential three-fidelity projections | Aggregate replay | Stored CNP outputs and active-learning rows are available |
| 100-trial HF coverage | Aggregate replay | Validation targets are complete; exact historical model state is absent |
| No-CNP ablation | Aggregate replay | Inputs are available; exact historical model state is absent |
| Table 1 optimum and acquisition surfaces | Unavailable exactly | Exact fitted surrogate/acquisition state and a complete initial set are absent |
| End-to-end CPU cost claim | Not recomputed | Simulation jobs and accounting records are not included |

A future exact reproduction package needs portable event-level LF and HF data, checksums, the original CNP checkpoint or a fully declared training recipe, the missing LF aggregate row, exact package versions, model noise constraints, optimizer seeds, and acquisition settings.